# Lesson 13: Tracing with OpenTelemetry (solution)

**Goal:** put spans around LLM calls and read the attributes a trace viewer like Phoenix shows.

**Before you start**
- Theory: [lesson 13](../../../docs/lessons/13_tracing.md)
- Python concepts: context managers (`with`), `__enter__` / `__exit__`
- Files used: `src/docrag/shared/tracing.py`, the fake LLM; spans are kept in memory (no Phoenix needed). Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import time

from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from docrag.llm.fake import FakeProvider
from docrag.llm.proxy import LLMProxy
from docrag.llm.types import Message
from docrag.shared import tracing

# The global tracer provider can be set once per process: only on the first run.
if "exporter" not in globals():
    exporter = InMemorySpanExporter()
    tracing.setup_tracing(None, exporter=exporter)
proxy = LLMProxy({"fake": FakeProvider()})
msgs = [Message(role="user", content="hello")]

## Step 1: A span around an LLM call

`@timed` functions (like the proxy's `chat`) only create spans **inside** a trace, so ingest stays out of the trace store. `tracing.root_span(name)` starts a trace: it's a context manager, everything called inside the `with` block becomes its child (even across `await`).

In [ ]:
exporter.clear()
with tracing.root_span("lesson13"):
    await proxy.chat("fake:m", msgs)

In [ ]:
spans = {s.name: s for s in exporter.get_finished_spans()}
assert "lesson13" in spans and "llm" in spans, (
    f"Expected a root and an llm span, got {list(spans)}"
)
assert spans["llm"].parent.span_id == spans["lesson13"].context.span_id, (
    "llm must be a child of the root"
)
await proxy.chat("fake:m", msgs)  # outside a trace: no new span
assert len(exporter.get_finished_spans()) == 2, (
    "@timed only joins a trace, it never starts one"
)
print({name: s.parent is None for name, s in spans.items()}, "(True = root)")
print("OK")

## Step 2: Your own context manager

A class with `__enter__` and `__exit__` works with `with` (Java's try-with-resources + `AutoCloseable`, Rust's `Drop`, but scoped). `__exit__` runs even when the block raises; returning `False` lets the exception continue. Write a `Stopwatch` that stores the elapsed seconds in `self.elapsed`.

In [ ]:
class Stopwatch:
    elapsed: float = 0.0

    def __enter__(self) -> "Stopwatch":
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc: object) -> bool:
        self.elapsed = time.perf_counter() - self._start
        return False

In [ ]:
try:
    with Stopwatch() as sw:
        sum(range(100_000))
except (TypeError, AttributeError) as err:
    raise AssertionError(f"Implement __enter__ and __exit__: {err}") from None
assert sw.elapsed > 0, "elapsed must be set in __exit__"
try:
    with Stopwatch() as sw2:
        raise KeyError("boom")
except KeyError:
    pass
else:
    raise AssertionError("__exit__ must return False so exceptions propagate")
assert sw2.elapsed > 0, "__exit__ runs even when the block raises"
print(f"{sw.elapsed * 1000:.2f} ms")
print("OK")

## Step 3: What's on the LLM span

Attributes follow the OpenInference conventions, so Phoenix can render prompts, tokens and costs. `span.attributes` behaves like a dict. Collect the attributes whose key starts with `"llm."`.

In [ ]:
llm_span = spans["llm"]
llm_attrs = {k: v for k, v in llm_span.attributes.items() if k.startswith("llm.")}

In [ ]:
assert llm_attrs.get("llm.model_name") == "m", "The model part of the route 'fake:m'"
assert llm_attrs.get("llm.token_count.prompt") == 100, (
    "FakeProvider reports 100 input tokens"
)
assert "llm.input_messages.0.message.content" in llm_attrs, "The prompt is on the span"
for k, v in sorted(llm_attrs.items()):
    print(f"{k:40} {str(v)[:50]}")
print(
    "docrag.cost_eur:",
    llm_span.attributes.get("docrag.cost_eur"),
    "| cache_hit:",
    llm_span.attributes.get("docrag.cache_hit"),
)
print("OK")

## Observation question

Which attributes do you really need in production?

<details>
<summary>Answer</summary>

The ones you filter, alert or pay on: model and provider, token counts, cost, latency (the span duration), cache hit, status/error, and an ID that ties traces together (here `session.id` = thread id). For RAG debugging also the retrieved chunk IDs and the Verify result. Full prompts and document text are the *expensive* part: they hold user questions and document content (PII, confidential data) and make spans large. In production you'd sample, redact or switch off content capture, and keep retention short.
</details>

## The real code

- Tracing helpers: [`src/docrag/shared/tracing.py`](../../../src/docrag/shared/tracing.py)
- @timed -> spans: [`src/docrag/shared/timing.py`](../../../src/docrag/shared/timing.py)
- Tests: [`tests/unit/test_tracing.py`](../../../tests/unit/test_tracing.py)
- Lesson: [`docs/lessons/13_tracing.md`](../../../docs/lessons/13_tracing.md)